# Zajęcia 8 — udostępnianie modelu: predykcja wsadowa i usługa

Notebook buduje usługę predykcyjną z jawnym kontraktem, testuje ją, mierzy opóźnienie i pokazuje wariant wsadowy oparty na tym samym kodzie scoringowym.

Usługa działa lokalnie, jako osobny proces na `127.0.0.1`. Na końcu notebooka jest zatrzymywana.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka i modelu

Trenujemy model i zapisujemy go jako artefakt. Usługa wczyta go z dysku — tak samo, jak zrobiłaby to po wdrożeniu. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import json
import os
import socket
import statistics
import subprocess
import sys
import textwrap
import time
from pathlib import Path

import httpx
import joblib
import pandas as pd
from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = REPO_ROOT / "artifacts" / "zajecia_08"
SRC_DIR = BASE_DIR / "src"
MODELS_DIR = BASE_DIR / "models"
REPORTS_DIR = BASE_DIR / "reports"
for directory in (SRC_DIR, MODELS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "model.joblib"
MODEL_VERSION = "1.0.0"
SEED = 42
FEATURE_COLUMNS = [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)",
]

frame = load_iris(as_frame=True).frame
x_train, x_test, y_train, y_test = train_test_split(
    frame[FEATURE_COLUMNS],
    frame["target"],
    test_size=0.3,
    random_state=SEED,
    stratify=frame["target"],
)
model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=500, random_state=SEED)),
    ]
).fit(x_train, y_train)
joblib.dump(model, MODEL_PATH)


def write_source(name: str, content: str) -> Path:
    path = SRC_DIR / name
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    return path


print(f"Model zapisany: {MODEL_PATH.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Wspólny kod scoringowy

Usługa i zadanie wsadowe muszą używać **tego samego** kodu przygotowania cech i predykcji. Inaczej wracamy do problemu rozjazdu ścieżek z zajęć 4.

Moduł `scoring.py` mapuje nazwy z API na nazwy kolumn modelu — interfejs publiczny nie musi znać wewnętrznych nazw. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
write_source(
    "scoring.py",
    """
    # Wspólny kod scoringowy dla usługi i zadania wsadowego.
    from __future__ import annotations

    import pandas as pd

    FEATURE_COLUMNS = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)",
    ]

    # Kontrakt API używa własnych nazw, niezależnych od nazw kolumn modelu.
    API_TO_MODEL = {
        "sepal_length_cm": "sepal length (cm)",
        "sepal_width_cm": "sepal width (cm)",
        "petal_length_cm": "petal length (cm)",
        "petal_width_cm": "petal width (cm)",
    }

    LABELS = ["setosa", "versicolor", "virginica"]


    def to_model_frame(records: list[dict]) -> pd.DataFrame:
        frame = pd.DataFrame.from_records(records).rename(columns=API_TO_MODEL)
        missing = sorted(set(FEATURE_COLUMNS) - set(frame.columns))
        if missing:
            raise ValueError(f"Brak wymaganych cech: {missing}")
        return frame[FEATURE_COLUMNS].astype(float)


    def score(model, records: list[dict]) -> list[dict]:
        predictions = model.predict(to_model_frame(records))
        return [
            {"prediction": int(value), "label": LABELS[int(value)]} for value in predictions
        ]
    """,
)
print("Zapisano scoring.py")

## 3. Usługa predykcyjna

Usługa ma cztery elementy poza samą predykcją: walidację żądania na brzegu, wersjonowaną ścieżkę, rozdzielone `health` i `ready` oraz strukturalny log bez danych wrażliwych.

**Zadanie:** uzupełnij aplikację o:

- ograniczenie liczby rekordów w żądaniu (od 1 do 100),
- zakresy dopuszczalnych wartości cech,
- endpoint `/ready` zwracający 503, gdy model nie jest wczytany,
- pola `contract_version`, `model_version` i `request_id` w odpowiedzi,
- logowanie metadanych żądania bez wartości cech.

In [ ]:
# Zadanie: dokończ usługę predykcyjną.
write_source(
    "app.py",
    """
    # Usługa predykcyjna.
    from __future__ import annotations

    import os
    from pathlib import Path

    import joblib
    from fastapi import FastAPI
    from pydantic import BaseModel

    from scoring import score

    CONTRACT_VERSION = "v1"
    MODEL_VERSION = os.environ.get("MODEL_VERSION", "0.0.0")
    MODEL = joblib.load(Path(os.environ["MODEL_PATH"]))

    app = FastAPI(title="Iris service", version=CONTRACT_VERSION)


    class IrisRecord(BaseModel):
        sepal_length_cm: float
        sepal_width_cm: float
        petal_length_cm: float
        petal_width_cm: float


    class PredictRequest(BaseModel):
        records: list[IrisRecord]


    @app.get("/health")
    def health() -> dict:
        return {"status": "ok"}


    @app.post("/v1/predict")
    def predict(request: PredictRequest) -> dict:
        return {"predictions": score(MODEL, [item.model_dump() for item in request.records])}
    """,
)
print("Zapisano app.py (wersja niepełna)")

In [ ]:
%%skip True
write_source(
    "app.py",
    """
    # Usługa predykcyjna z jawnym kontraktem.
    from __future__ import annotations

    import json
    import logging
    import os
    import time
    import uuid
    from pathlib import Path

    import joblib
    from fastapi import FastAPI, Response
    from pydantic import BaseModel, Field

    from scoring import score

    CONTRACT_VERSION = "v1"
    MODEL_VERSION = os.environ.get("MODEL_VERSION", "0.0.0")
    MAX_RECORDS = 100

    logging.basicConfig(level=logging.INFO, format="%(message)s")
    logger = logging.getLogger("service")

    MODEL = None
    try:
        MODEL = joblib.load(Path(os.environ["MODEL_PATH"]))
    except Exception as error:  # usługa wstaje, ale nie przyjmuje ruchu
        logger.error(json.dumps({"event": "model_load_failed", "error": str(error)}))

    app = FastAPI(title="Iris service", version=CONTRACT_VERSION)


    class IrisRecord(BaseModel):
        sepal_length_cm: float = Field(ge=0.0, le=20.0)
        sepal_width_cm: float = Field(ge=0.0, le=20.0)
        petal_length_cm: float = Field(ge=0.0, le=20.0)
        petal_width_cm: float = Field(ge=0.0, le=20.0)


    class PredictRequest(BaseModel):
        records: list[IrisRecord] = Field(min_length=1, max_length=MAX_RECORDS)


    class Prediction(BaseModel):
        prediction: int
        label: str


    class PredictResponse(BaseModel):
        contract_version: str
        model_version: str
        request_id: str
        predictions: list[Prediction]


    @app.get("/health")
    def health() -> dict:
        return {"status": "alive", "contract_version": CONTRACT_VERSION}


    @app.get("/ready")
    def ready(response: Response) -> dict:
        if MODEL is None:
            response.status_code = 503
            return {"status": "model_not_loaded"}
        return {"status": "ready", "model_version": MODEL_VERSION}


    @app.post("/v1/predict", response_model=PredictResponse)
    def predict(request: PredictRequest, response: Response) -> dict:
        request_id = str(uuid.uuid4())
        started = time.perf_counter()

        predictions = score(MODEL, [item.model_dump() for item in request.records])
        duration_ms = (time.perf_counter() - started) * 1000

        # Log zawiera wyłącznie metadane — żadnych wartości cech.
        logger.info(
            json.dumps(
                {
                    "event": "prediction",
                    "request_id": request_id,
                    "records": len(request.records),
                    "duration_ms": round(duration_ms, 2),
                    "model_version": MODEL_VERSION,
                    "contract_version": CONTRACT_VERSION,
                }
            )
        )

        return {
            "contract_version": CONTRACT_VERSION,
            "model_version": MODEL_VERSION,
            "request_id": request_id,
            "predictions": predictions,
        }
    """,
)
print("Zapisano app.py (wersja pełna)")

## 4. Uruchomienie usługi

Usługę uruchamiamy jako **osobny proces**, tak jak zrobiłby to system operacyjny po wdrożeniu. Notebook jest tylko klientem. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
def free_port() -> int:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]


PORT = free_port()
BASE_URL = f"http://127.0.0.1:{PORT}"

environment = os.environ.copy()
environment["MODEL_PATH"] = str(MODEL_PATH)
environment["MODEL_VERSION"] = MODEL_VERSION


def start_service() -> subprocess.Popen:
    process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "app:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(PORT),
            "--log-level",
            "warning",
        ],
        cwd=SRC_DIR,
        env=environment,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(60):
        try:
            if httpx.get(f"{BASE_URL}/health", timeout=1.0).status_code == 200:
                return process
        except Exception:
            time.sleep(0.5)
    process.kill()
    raise RuntimeError("Usługa nie wstała w wyznaczonym czasie.")


def restart_service(process: subprocess.Popen) -> subprocess.Popen:
    process.terminate()
    try:
        process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        process.kill()
    return start_service()


server = start_service()
print(f"Usługa działa pod adresem {BASE_URL}")
print(f"/health -> {httpx.get(f'{BASE_URL}/health').json()}")
print(f"/ready  -> {httpx.get(f'{BASE_URL}/ready').json()}")

## 5. Test kontraktu

Test kontraktu sprawdza **umowę**, a nie jakość modelu. Ma wykryć każdą zmianę, która zepsułaby integrację u konsumenta.

**Zadanie:** napisz funkcję `contract_test()`, która sprawdza: kod 200 dla poprawnego żądania, obecność wszystkich pól odpowiedzi, typ predykcji, zgodność liczby predykcji z liczbą rekordów oraz kod 422 dla żądania niezgodnego z kontraktem.

In [ ]:
SAMPLE = [
    {
        "sepal_length_cm": 5.1,
        "sepal_width_cm": 3.5,
        "petal_length_cm": 1.4,
        "petal_width_cm": 0.2,
    },
    {
        "sepal_length_cm": 6.7,
        "sepal_width_cm": 3.0,
        "petal_length_cm": 5.2,
        "petal_width_cm": 2.3,
    },
]

# Zadanie: napisz test kontraktu usługi.


In [ ]:
%%skip True
REQUIRED_RESPONSE_KEYS = {"contract_version", "model_version", "request_id", "predictions"}


def contract_test() -> dict:
    response = httpx.post(f"{BASE_URL}/v1/predict", json={"records": SAMPLE}, timeout=10.0)
    assert response.status_code == 200, response.status_code

    body = response.json()
    assert set(body) == REQUIRED_RESPONSE_KEYS, f"pola odpowiedzi: {sorted(body)}"
    assert body["contract_version"] == "v1"
    assert len(body["predictions"]) == len(SAMPLE)
    for item in body["predictions"]:
        assert set(item) == {"prediction", "label"}, sorted(item)
        assert isinstance(item["prediction"], int)

    broken = httpx.post(f"{BASE_URL}/v1/predict", json={"records": [{}]}, timeout=10.0)
    assert broken.status_code == 422, broken.status_code

    print("Test kontraktu przeszedł.")
    return body


display(contract_test())

## 6. Punkt kontrolny 1 — żądania niezgodne z kontraktem

Wysyłamy pięć rodzajów wadliwych żądań. Walidacja odbywa się na brzegu usługi, więc model w ogóle ich nie widzi.

**Do opisania:** który błąd zostałby wykryty dopiero przez model, gdyby nie było walidacji na brzegu, a który nie zostałby wykryty wcale?

In [ ]:
cases = {
    "brak pola": {"records": [{k: v for k, v in SAMPLE[0].items() if k != "petal_width_cm"}]},
    "zły typ": {"records": [{**SAMPLE[0], "petal_width_cm": "szeroki"}]},
    "wartość spoza zakresu": {"records": [{**SAMPLE[0], "sepal_length_cm": 999.0}]},
    "pusta lista rekordów": {"records": []},
    "za dużo rekordów": {"records": SAMPLE * 60},
}

rows = []
for name, payload in cases.items():
    response = httpx.post(f"{BASE_URL}/v1/predict", json=payload, timeout=10.0)
    detail = response.json().get("detail", [])
    message = detail[0]["msg"] if isinstance(detail, list) and detail else "-"
    rows.append({"przypadek": name, "kod": response.status_code, "komunikat": message[:60]})

display(pd.DataFrame(rows))

## 7. Punkt kontrolny 2 — złamanie kontraktu odpowiedzi

Usuwamy pole `model_version` z odpowiedzi, restartujemy usługę i uruchamiamy test kontraktu.

**Do opisania:** co zobaczyłby konsument, gdyby testu nie było, i w którym momencie by to zauważył?

In [ ]:
APP_PATH = SRC_DIR / "app.py"
original_app = APP_PATH.read_text(encoding="utf-8")

broken_app = original_app.replace(
    "    contract_version: str\n    model_version: str\n", "    contract_version: str\n"
).replace('        "model_version": MODEL_VERSION,\n', "")
assert broken_app != original_app, "Podmiana nie zadziałała — sprawdź treść app.py."

APP_PATH.write_text(broken_app, encoding="utf-8")
server = restart_service(server)

response = httpx.post(f"{BASE_URL}/v1/predict", json={"records": SAMPLE[:1]}, timeout=10.0)
print(f"Odpowiedź po zmianie: {sorted(response.json())}\n")
try:
    contract_test()
except AssertionError as error:
    print(f"Test kontraktu wykrył zmianę łamiącą umowę: {error}")

APP_PATH.write_text(original_app, encoding="utf-8")
server = restart_service(server)
print()
contract_test()

## 8. Punkt kontrolny 3 — opóźnienie w percentylach

Średnia nie opisuje doświadczenia użytkownika. Mierzymy p50, p95 i p99 dla żądań pojedynczych oraz koszt jednego rekordu w żądaniu zbiorczym.

**Zadanie:** zmierz opóźnienie 200 żądań pojedynczych i 20 żądań zbiorczych po 50 rekordów. Policz percentyle i koszt na rekord, a wynik zapisz do `reports/latency.json`.

In [ ]:
# Zadanie: zmierz opóźnienie i zapisz raport.
# Wskazówka: statistics.quantiles(dane, n=100) zwraca 99 punktów podziału.


In [ ]:
%%skip True
def percentiles(samples: list) -> dict:
    cuts = statistics.quantiles(samples, n=100)
    return {
        "p50_ms": round(statistics.median(samples), 2),
        "p95_ms": round(cuts[94], 2),
        "p99_ms": round(cuts[98], 2),
        "mean_ms": round(statistics.fmean(samples), 2),
    }


with httpx.Client(base_url=BASE_URL, timeout=10.0) as http:
    http.post("/v1/predict", json={"records": SAMPLE})  # rozgrzewka

    single = []
    for _ in range(200):
        started = time.perf_counter()
        http.post("/v1/predict", json={"records": SAMPLE[:1]})
        single.append((time.perf_counter() - started) * 1000)

    batch = []
    for _ in range(20):
        started = time.perf_counter()
        http.post("/v1/predict", json={"records": SAMPLE * 25})
        batch.append((time.perf_counter() - started) * 1000)

latency = {
    "pojedyncze_zadania": percentiles(single),
    "zadania_zbiorcze_50_rekordow": percentiles(batch),
    "koszt_na_rekord_pojedynczo_ms": round(statistics.median(single), 3),
    "koszt_na_rekord_zbiorczo_ms": round(statistics.median(batch) / 50, 3),
}
latency["ile_razy_taniej_zbiorczo"] = round(
    latency["koszt_na_rekord_pojedynczo_ms"] / latency["koszt_na_rekord_zbiorczo_ms"], 1
)

(REPORTS_DIR / "latency.json").write_text(json.dumps(latency, indent=2), encoding="utf-8")
print(json.dumps(latency, indent=2))

## 9. Wariant wsadowy

To samo zadanie bez usługi: wejście z pliku, wyjście do pliku, raport wykonania. Kod scoringowy jest ten sam, więc obie ścieżki nie mogą się rozjechać.

**Zadanie:** napisz `batch.py`, który wczytuje CSV, odrzuca rekordy niepoprawne (raportując ich liczbę zamiast przerywać), zapisuje predykcje oraz raport z liczbą rekordów, liczbą odrzuconych, czasem i wersją modelu. Uruchom go dwa razy i sprawdź idempotencję.

In [ ]:
batch_input = REPORTS_DIR / "batch_input.csv"
input_frame = x_test.rename(
    columns={
        "sepal length (cm)": "sepal_length_cm",
        "sepal width (cm)": "sepal_width_cm",
        "petal length (cm)": "petal_length_cm",
        "petal width (cm)": "petal_width_cm",
    }
).copy()
input_frame.iloc[0, input_frame.columns.get_loc("petal_width_cm")] = float("nan")
input_frame.to_csv(batch_input, index=False)
print(f"Wejście wsadowe: {len(input_frame)} rekordów, w tym jeden niekompletny.")

# Zadanie: napisz i uruchom zadanie wsadowe.


In [ ]:
%%skip True
write_source(
    "batch.py",
    """
    # Niezawodne zadanie wsadowe korzystające z tego samego kodu scoringowego.
    from __future__ import annotations

    import argparse
    import json
    import os
    import time
    from pathlib import Path

    import joblib
    import pandas as pd

    from scoring import API_TO_MODEL, score

    MODEL_VERSION = os.environ.get("MODEL_VERSION", "0.0.0")


    def main() -> None:
        parser = argparse.ArgumentParser(description="Predykcja wsadowa")
        parser.add_argument("--input", required=True)
        parser.add_argument("--output", required=True)
        parser.add_argument("--report", required=True)
        args = parser.parse_args()

        started = time.perf_counter()
        model = joblib.load(Path(os.environ["MODEL_PATH"]))
        frame = pd.read_csv(args.input)

        required = list(API_TO_MODEL)
        complete = frame.dropna(subset=required)
        rejected = len(frame) - len(complete)

        predictions = score(model, complete[required].to_dict(orient="records"))
        result = complete.copy()
        result["prediction"] = [item["prediction"] for item in predictions]
        result["label"] = [item["label"] for item in predictions]
        result.to_csv(args.output, index=False)

        report = {
            "input_rows": int(len(frame)),
            "scored_rows": int(len(complete)),
            "rejected_rows": int(rejected),
            "model_version": MODEL_VERSION,
            "duration_s": round(time.perf_counter() - started, 3),
        }
        Path(args.report).write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(json.dumps(report, indent=2))


    if __name__ == "__main__":
        main()
    """,
)

predictions_path = REPORTS_DIR / "predictions.csv"
report_path = REPORTS_DIR / "batch_report.json"


def run_batch() -> str:
    result = subprocess.run(
        [
            sys.executable,
            "batch.py",
            "--input",
            str(batch_input),
            "--output",
            str(predictions_path),
            "--report",
            str(report_path),
        ],
        cwd=SRC_DIR,
        env=environment,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    print(result.stdout)
    return predictions_path.read_text(encoding="utf-8")


first = run_batch()
second = run_batch()
print(f"Dwa uruchomienia dały identyczny wynik: {first == second}")
display(pd.read_csv(predictions_path).head())

## 10. Zatrzymanie usługi

Proces usługi trzeba zamknąć, inaczej zostanie w tle po zakończeniu pracy z notebookiem. Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
server.terminate()
try:
    server.wait(timeout=15)
except subprocess.TimeoutExpired:
    server.kill()
print(f"Usługa zatrzymana (kod wyjścia: {server.returncode}).")

## 11. Podsumowanie

Model jest wywoływalny z zewnątrz i ma jawny, wersjonowany kontrakt. Ten sam kod scoringowy obsługuje ścieżkę online i wsadową, więc nie mogą się rozjechać. Wiemy też, ile kosztuje pojedyncza predykcja.

Czego nadal brakuje:

- usługa działa tylko na tym komputerze, bez przenośnego artefaktu wykonawczego (zajęcia 9);
- nikt nie uruchamia testów kontraktu automatycznie przy zmianie kodu (zajęcia 9);
- nie ma bezpiecznej procedury zmiany wersji na produkcji ani wycofania (zajęcia 10);
- logi powstają, ale nikt ich nie zbiera ani nie obserwuje (zajęcia 11).

### Zadanie do własnego projektu

1. Udostępnienie modelu jako usługi HTTP albo niezawodnego zadania wsadowego, z uzasadnieniem wyboru.
2. Jawny, wersjonowany kontrakt; nazwy w API niezależne od nazw kolumn modelu.
3. Test kontraktu dołączony do zestawu testów z zajęć 6.
4. Rozdzielone sprawdzenie stanu: `health` i `ready`.
5. Logowanie strukturalne z identyfikatorem żądania, bez danych wrażliwych.
6. Pomiar opóźnienia w percentylach albo czasu przetworzenia paczki, zapisany jako artefakt.
7. Opis skutków godzinnej niedostępności usługi.